# Probabilistic Diagnostic Reasoning Demo (v2)
This notebook runs a working loop for:
- one error code + user symptoms
- optional warnings/counters (manual for now)
- Bayesian updates (Naive Bayes)
- next-step suggestion via expected information gain with soft cost penalty
- **dynamic upstream cause expansion**
- **confirm-test required for expensive repairs** (parts cost threshold)

Logs each session to `logs/diagnostic_sessions.jsonl`.


In [ ]:
import json, math, uuid, datetime
from pathlib import Path

KB_PATH = Path(r"/mnt/data/kb_ink_system_pump_tube_v3.json")
kb = json.loads(KB_PATH.read_text(encoding="utf-8"))

root_causes = {rc["root_cause_id"]: rc for rc in kb["catalogs"]["root_causes"]}
obs_catalog = {o["observation_id"]: o for o in kb["catalogs"]["observations"]}
procedures = {p["procedure_id"]: p for p in kb["catalogs"]["procedures"]}
error_models = {m["error_code"]: m for m in kb["catalogs"]["error_code_models"]}

likelihoods = kb["parameters"].get("likelihoods", {})
action_success = kb["parameters"].get("action_success", {})

cfg = kb["inference_config"]
EPSILON = cfg["stop_condition"]["epsilon_default"]
REPAIR_RECOMMEND = cfg["repair_recommendation"]["recommend_threshold"]
REPAIR_CONSIDER = cfg["repair_recommendation"]["consider_threshold"]

UP_CFG = cfg.get("upstream_generation", {})
EXP_CFG = cfg.get("expensive_repair_policy", {})
EXP_PARTS_TH = EXP_CFG.get("parts_eur_threshold", 500)
CONFIRM_MAP = EXP_CFG.get("confirming_procedures_by_repair", {})
CONFIRM_FALLBACK = EXP_CFG.get("fallback_if_no_mapping", "any_check_or_test")

print("Loaded KB:", kb["meta"]["name"])
print("Error codes:", len(error_models), "| Causes:", len(root_causes), "| Procedures:", len(procedures))


In [ ]:
import re

def slug(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def ensure_symptom_observation(symptom: str) -> str:
    oid = "OBS_SYM_" + slug(symptom).upper()[:50]
    if oid not in obs_catalog:
        obs_catalog[oid] = {
            "observation_id": oid,
            "label": f"Symptom: {symptom}",
            "type": "binary",
            "values": ["yes", "no"]
        }
    return oid

def add_placeholder_symptom_likelihoods(error_code: str):
    model = error_models[error_code]
    listed_causes = set(model["candidate_root_causes"])
    for sym in model.get("symptoms", []):
        oid = ensure_symptom_observation(sym)
        like = likelihoods.setdefault(oid, {})
        for cid in root_causes.keys():
            if cid in like:
                continue
            p_yes = 0.60 if cid in listed_causes else 0.52
            like[cid] = {"yes": p_yes, "no": 1.0 - p_yes}


In [ ]:
def normalize(dist: dict) -> dict:
    s = sum(dist.values())
    if s <= 0:
        n = len(dist)
        return {k: 1.0/n for k in dist} if n else {}
    return {k: v/s for k, v in dist.items()}

def entropy(dist: dict) -> float:
    H = 0.0
    for p in dist.values():
        if p > 1e-12:
            H -= p * math.log(p, 2)
    return H

def get_likelihood(obs_id: str, cause_id: str, value: str) -> float:
    vals = obs_catalog[obs_id]["values"]
    table = likelihoods.get(obs_id, {}).get(cause_id)
    if table is None:
        return 1.0 / len(vals)
    return float(table.get(value, 1.0 / len(vals)))

def update_belief(belief: dict, obs_id: str, value: str) -> dict:
    updated = {}
    for c, p in belief.items():
        updated[c] = p * get_likelihood(obs_id, c, value)
    return normalize(updated)

def action_outcome_likelihood(action_id: str, cause_id: str, resolved_value: str) -> float:
    table = action_success.get(action_id, {}).get(cause_id)
    if table is None:
        return 0.45 if resolved_value == "yes" else 0.55
    return float(table.get(resolved_value, 0.5))

def update_after_repair(belief: dict, action_id: str, resolved_value: str) -> dict:
    updated = {}
    for c, p in belief.items():
        updated[c] = p * action_outcome_likelihood(action_id, c, resolved_value)
    return normalize(updated)

def top_k(belief: dict, k: int = 10):
    return sorted(belief.items(), key=lambda kv: kv[1], reverse=True)[:k]


In [ ]:
def context_keywords(error_code: str, symptoms_yes: list[str], warnings: dict) -> set[str]:
    model = error_models[error_code]
    txt = (model.get("description","") + " " + " ".join(symptoms_yes)).lower()
    # include warning ids that are yes
    for oid, val in warnings.items():
        if val == "yes":
            txt += " " + oid.lower()
    keys = set()
    for k in UP_CFG.get("keywords", []):
        if k in txt:
            keys.add(k)
    return keys

def propose_upstream_causes(error_code: str, symptoms_yes: list[str], warnings: dict, max_n: int = 15) -> list[str]:
    listed = set(error_models[error_code]["candidate_root_causes"])
    keys = context_keywords(error_code, symptoms_yes, warnings)

    scored = []
    for cid, rc in root_causes.items():
        if cid in listed:
            continue
        label = (rc.get("label","") or "").lower()
        tags = set(rc.get("tags") or [])
        score = 0
        for k in keys:
            if k in label:
                score += 2
            if k.replace(" ", "_") in tags:
                score += 2
        # also: if warning 290005 is yes, boost pump_tube-tagged causes
        if warnings.get("OBS_WARNING_290005_SEEN") == "yes" and "pump_tube" in tags:
            score += 3
        if warnings.get("OBS_WARNING_290020_SEEN") == "yes" and "ink_expired" in tags:
            score += 3
        if score > 0:
            scored.append((score, cid))

    scored.sort(reverse=True)
    return [cid for _, cid in scored[:max_n]]

def initial_belief(error_code: str, symptoms_yes: list[str], warnings: dict, alpha: float = 0.90) -> dict:
    ec_prior = normalize({cid: float(p) for cid, p in error_models[error_code]["priors_root_cause_given_error"].items()})
    belief = {cid: 0.0 for cid in root_causes.keys()}

    upstream = propose_upstream_causes(error_code, symptoms_yes, warnings, max_n=UP_CFG.get("max_upstream_causes",15))
    other_mass = max(0.0, 1.0 - alpha)
    if upstream:
        base = other_mass / len(upstream)
        for cid in upstream:
            belief[cid] += base
    else:
        # fallback: spread over all non-listed causes
        other = [cid for cid in root_causes.keys() if cid not in ec_prior]
        if other:
            base = other_mass / len(other)
            for cid in other:
                belief[cid] += base

    for cid, p in ec_prior.items():
        belief[cid] += alpha * p

    return normalize(belief), upstream


In [ ]:
def predicted_outcome_distribution(belief: dict, proc_id: str) -> dict:
    proc = procedures[proc_id]
    vals = proc["outcomes"]
    dist = {v: 0.0 for v in vals}

    if proc["kind"] == "repair":
        for v in vals:
            for c, p in belief.items():
                dist[v] += p * action_outcome_likelihood(proc_id, c, v)
    else:
        obs_id = proc["produces_observation"]
        for v in vals:
            for c, p in belief.items():
                dist[v] += p * get_likelihood(obs_id, c, v)

    return normalize(dist)

def expected_entropy_after(belief: dict, proc_id: str) -> float:
    proc = procedures[proc_id]
    out_dist = predicted_outcome_distribution(belief, proc_id)

    expH = 0.0
    for v, pv in out_dist.items():
        if pv <= 1e-12:
            continue
        if proc["kind"] == "repair":
            post = update_after_repair(belief, proc_id, v)
        else:
            post = update_belief(belief, proc["produces_observation"], v)
        expH += pv * entropy(post)
    return expH

def score_procedure(belief: dict, proc_id: str, lambda_time: float = 0.02, lambda_parts: float = 0.005) -> dict:
    H0 = entropy(belief)
    expH = expected_entropy_after(belief, proc_id)
    ig = H0 - expH
    cost = procedures[proc_id]["cost"]
    score = ig - lambda_time * cost["time_min"] - lambda_parts * cost["parts_eur"]
    return {"procedure_id": proc_id, "score": score, "ig": ig, "time_min": cost["time_min"], "parts_eur": cost["parts_eur"]}

def confirm_done_for(repair_id: str, tried: set) -> bool:
    required = CONFIRM_MAP.get(repair_id)
    if required:
        return any(pid in tried for pid in required)
    if CONFIRM_FALLBACK == "any_check_or_test":
        return any(procedures[pid]["kind"] in ("check","test") for pid in tried)
    return False

def suggest_next(belief: dict, error_code: str, tried: set, lambda_time=0.02, lambda_parts=0.005) -> list:
    cand = error_models[error_code]["candidate_procedure_ids"]
    scored = []
    for pid in cand:
        if pid in tried:
            continue
        proc = procedures[pid]
        # expensive repair gate
        if proc["kind"] == "repair" and proc.get("is_expensive", False):
            if not confirm_done_for(pid, tried):
                continue
        scored.append(score_procedure(belief, pid, lambda_time=lambda_time, lambda_parts=lambda_parts))
    return sorted(scored, key=lambda d: d["score"], reverse=True)


In [ ]:
from pathlib import Path

def exhausted(belief: dict, epsilon: float = EPSILON) -> bool:
    return all(p < epsilon for p in belief.values())

def run_session(
    error_code: str,
    symptoms_yes=None,
    warnings=None,
    counters=None,
    max_steps: int = 15,
    log_dir: str = "logs",
    lambda_time: float = 0.02,
    lambda_parts: float = 0.005
):
    if error_code not in error_models:
        raise ValueError(f"Unknown error code: {error_code}")

    symptoms_yes = symptoms_yes or []
    warnings = warnings or {}
    counters = counters or {}

    add_placeholder_symptom_likelihoods(error_code)

    session_id = str(uuid.uuid4())
    ts = datetime.datetime.utcnow().isoformat() + "Z"
    tried = set()
    history = []

    belief, upstream = initial_belief(error_code, symptoms_yes, warnings, alpha=UP_CFG.get("alpha_error_code_mass",0.90))

    # initial observations
    for s in symptoms_yes:
        oid = ensure_symptom_observation(s)
        belief = update_belief(belief, oid, "yes")
        history.append({"type":"symptom","observation_id":oid,"value":"yes","note":s})

    for obs_id, val in warnings.items():
        belief = update_belief(belief, obs_id, val)
        history.append({"type":"warning","observation_id":obs_id,"value":val})

    for obs_id, val in counters.items():
        belief = update_belief(belief, obs_id, val)
        history.append({"type":"counter","observation_id":obs_id,"value":val})

    print("\nSESSION", session_id, "| error_code", error_code)
    print("epsilon:", EPSILON, "| recommend:", REPAIR_RECOMMEND, "| consider:", REPAIR_CONSIDER)
    if upstream:
        print("Dynamic upstream causes added:", len(upstream))

    last_repair_recommendation = None  # store top cause when a repair is actually executed

    for step in range(1, max_steps + 1):
        print("\n--- Step", step, "---")
        top = top_k(belief, 8)
        for cid, p in top:
            print(f"{p:0.3f}  {cid}  | {root_causes[cid]['label']}")

        if exhausted(belief):
            print("\nAll hypotheses < epsilon. ESCALATE.")
            history.append({"type":"stop","reason":"exhausted"})
            break

        suggestions = suggest_next(belief, error_code, tried, lambda_time=lambda_time, lambda_parts=lambda_parts)
        if not suggestions:
            print("\nNo more allowed procedures (expensive repairs gated). ESCALATE.")
            history.append({"type":"stop","reason":"no_procedures"})
            break

        best = suggestions[0]
        pid = best["procedure_id"]
        proc = procedures[pid]

        # if we are about to execute a repair and top prob is high, record the model's hypothesis for logging
        top_c, top_p = top[0]
        if proc["kind"] == "repair" and top_p >= REPAIR_CONSIDER:
            last_repair_recommendation = {
                "recommended_top_cause_id": top_c,
                "recommended_top_cause_p": top_p,
                "recommended_top_cause_label": root_causes[top_c]["label"],
                "repair_procedure_id": pid,
                "repair_label": proc["label"]
            }

        print(f"\nNEXT: {pid} | {proc['label']}")
        print(f"  kind={proc['kind']} score={best['score']:.4f} IG={best['ig']:.4f} time={best['time_min']}m parts=€{best['parts_eur']}")
        print("  outcomes:", proc["outcomes"])
        if proc["kind"] == "repair" and proc.get("is_expensive", False):
            print("  NOTE: expensive repair (confirm test required).")

        outcome = input("Enter outcome value: ").strip()
        if outcome not in proc["outcomes"]:
            print("Invalid outcome. Try again.")
            continue

        tried.add(pid)

        if proc["kind"] == "repair":
            belief = update_after_repair(belief, pid, outcome)
            history.append({"type":"repair","procedure_id":pid,"outcome":outcome, "recommended_snapshot": last_repair_recommendation})
            if outcome == "yes":
                print("\nResolved=yes. STOP.")
                history.append({"type":"stop","reason":"resolved"})
                break
        else:
            obs_id = proc["produces_observation"]
            belief = update_belief(belief, obs_id, outcome)
            history.append({"type":proc["kind"],"procedure_id":pid,"observation_id":obs_id,"value":outcome})

    # ground truth logging rule (your policy):
    # if a repair resolved, we log the root cause that was recommended when that repair was executed
    ground_truth = None
    if history and history[-1].get("reason") == "resolved":
        # search last repair entry with outcome yes
        for h in reversed(history):
            if h.get("type") == "repair" and h.get("outcome") == "yes":
                snap = h.get("recommended_snapshot")
                if snap:
                    ground_truth = {
                        "mode": "inferred_from_resolving_repair_recommendation",
                        "root_cause_id": snap["recommended_top_cause_id"],
                        "label": snap["recommended_top_cause_label"],
                        "p_at_recommendation": snap["recommended_top_cause_p"],
                        "repair_procedure_id": snap["repair_procedure_id"],
                        "repair_label": snap["repair_label"]
                    }
                break

    log_entry = {
        "session_id": session_id,
        "timestamp_utc": ts,
        "error_code": error_code,
        "symptoms_yes": symptoms_yes,
        "initial_warnings": warnings,
        "initial_counters": counters,
        "dynamic_upstream_causes": upstream,
        "history": history,
        "final_top_causes": [{"cause_id": cid, "p": p, "label": root_causes[cid]["label"]} for cid, p in top_k(belief, 10)],
        "epsilon": EPSILON,
        "lambda_time": lambda_time,
        "lambda_parts": lambda_parts,
        "ground_truth": ground_truth
    }

    Path(log_dir).mkdir(parents=True, exist_ok=True)
    log_path = Path(log_dir) / "diagnostic_sessions.jsonl"
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

    print("\nLog saved to:", log_path.resolve())
    return log_entry


## Example run
Edit inputs and re-run the next cell.

Useful observation ids:
- `OBS_WARNING_290005_SEEN`: "yes"/"no"
- `OBS_WARNING_290020_SEEN`: "yes"/"no"
- `OBS_LIFE_INK_PUMP`: "low"/"medium"/"high"
- `OBS_LIFE_PUMP_TUBE`: "low"/"medium"/"high"


In [ ]:
error_code = "250001"
symptoms_yes = ["Ink delivery to printhead failed"]
warnings = {"OBS_WARNING_290005_SEEN":"yes", "OBS_WARNING_290020_SEEN":"no"}
counters = {"OBS_LIFE_PUMP_TUBE":"high", "OBS_LIFE_INK_PUMP":"medium"}

log_entry = run_session(
    error_code,
    symptoms_yes=symptoms_yes,
    warnings=warnings,
    counters=counters,
    max_steps=10,
    lambda_time=0.02,
    lambda_parts=0.005
)


## Next: what I need from you to make v3 strong (not just a demo)

1) For each **expensive repair**, list the **confirming tests** that count.
   - For printhead replacement, which checks should be confirming?
   - (We used placeholders for now.)

2) Decide how “dynamic upstream” should propose causes:
   - Should it propose from **this JSON only** (recommended for v1), or also from other manuals/KBs?

3) Give a first guess for counter thresholds (later replaced by real data).
